# EXIOBASE import

## 1. Setup background

In [1]:
import bw2data as bd
import bw2io as bi
import pandas as pd

from functions.fun_exiobase_import import (
    get_location_from_name,
    production_as_datasetname,
    remove_empty_activities,
    remove_location_from_name,
    remove_old_ef,
    tweak3,
)
from functions.utils import Config

In [ ]:
bd.projects.set_current(Config.PROJECT_NAME)

print(f"Current Brightway project: {bd.projects.current}")

## 2. Import biosphere database

The EXIOBASE import is linked to the standard Brightway `biosphere3` database.

In [3]:
if Config.EXIOBASE_BIOSPHERE_NAME not in bd.databases:
    bi.create_default_biosphere3()

## 3. Create the EXIOBASE importer

EXIOBASE is imported from the SimaPro CSV file configured in `functions/utils.py`.

In [4]:
importer = bi.SimaProBlockCSVImporter(
    Config.EXIOBASE_CSV_PATH,
    database_name=Config.EXIOBASE_DATABASE_NAME,
    biosphere_database_name=Config.EXIOBASE_BIOSPHERE_NAME,
    separate_products=False,
)

2026-09-22 14:09:20.706 | INFO     | bw_simapro_csv.main:__init__:119 - Writing logs to C:\Users\SimonVemmelund\AppData\Local\pylca\bw_simapro_csv\Logs\exiobase_v3.3.16b2-2026-09-22T14-09-20
2026-09-22 14:09:20.720 | INFO     | bw_simapro_csv.main:__init__:142 - Using database name 'exiobase3316b2'
2026-09-22 14:09:20.722 | INFO     | bw_simapro_csv.main:__init__:147 - SimaPro CSV import started.
	File: 'C:\Users\SimonVemmelund\OneDrive - 2.-0 LCA Consultants ApS\Databases\EXIOBASE_BW\exiobase_v3.3.16b2.CSV'
	Delimiter: ';'
	Name: 'EXIOBASE v3316b2 re-structured'


2026-09-22 14:11:14.449 | INFO     | bw_simapro_csv.main:resolve_parameters:324 - Extracted and cleaned 38644 process datasets
2026-09-22 14:16:33.924 | WARNING  | bw_simapro_csv.units:normalize_units:112 - Unknown unit MEUR2011 used on line 440
2026-09-22 14:16:33.926 | WARNING  | bw_simapro_csv.units:normalize_units:112 - Unknown unit ha a used on line 675
2026-09-22 14:16:33.928 | WARNING  | bw_simapro_csv.units:normalize_units:112 - Unknown unit TJ used on line 764
2026-09-22 14:16:33.930 | WARNING  | bw_simapro_csv.units:normalize_units:112 - Unknown unit 1000 pers used on line 812
2026-09-22 14:16:33.932 | WARNING  | bw_simapro_csv.units:normalize_units:112 - Unknown unit million h used on line 819
2026-09-22 14:16:33.933 | WARNING  | bw_simapro_csv.units:normalize_units:112 - Unknown unit MEUR,PPP used on line 826
2026-09-22 14:16:33.934 | WARNING  | bw_simapro_csv.units:normalize_units:112 - Unknown unit n used on line 839
2026-09-22 14:16:36.856 | WARNING  | bw_simapro_csv.uni

## 4. Apply standard and EXIOBASE-specific strategies

Brightway first applies the standard SimaPro import strategies. The additional helper strategies then clean EXIOBASE activity names, locations, old elementary flows, and production exchanges.

The detailed helper functions are available in `functions/fun_exiobase_import.py`.

In [5]:
importer.apply_strategies()

importer.apply_strategy(get_location_from_name)
importer.apply_strategy(remove_location_from_name)
importer.apply_strategy(remove_old_ef)
importer.apply_strategy(production_as_datasetname)

# Check and normalize dataset units
print("Dataset units before normalization:")
print({ds["unit"] for ds in importer.data})

importer.apply_strategy(bi.strategies.generic.normalize_units)

print("Dataset units after normalization:")
print({ds["unit"] for ds in importer.data})

importer.apply_strategy(remove_empty_activities)

Applying strategy: set_metadata_using_single_functional_exchange
Applying strategy: override_process_name_using_single_functional_exchange
Applying strategy: drop_unspecified_subcategories
Applying strategy: split_simapro_name_geo
Applying strategy: create_products_as_new_nodes
Applying strategy: link_technosphere_based_on_name_unit_location
Applying strategy: link_iterable_by_fields
Applying strategy: match_internal_simapro_simapro_with_unit_conversion
Applied 8 strategies in 56.37 seconds
Applying strategy: get_location_from_name


Applying strategy: remove_location_from_name
Applying strategy: remove_old_ef
Applying strategy: production_as_datasetname
Dataset units before normalization:
{'p', 'kg', 'ha a', 'MEUR2011', 'TJ'}
Applying strategy: normalize_units
Dataset units after normalization:
{'unit', 'ha a', 'kilogram', 'MEUR2011', 'TJ'}
Applying strategy: remove_empty_activities


### 3a. Check exchange units

Inspect the units used by the imported exchanges after unit normalization.


In [6]:
exchange_units = {
    exchange["unit"]
    for dataset in importer.data
    for exchange in dataset["exchanges"]
}

sorted(exchange_units)

['1000 pers',
 'MEUR,PPP',
 'MEUR2011',
 'TJ',
 'ha a',
 'kilogram',
 'million h',
 'n',
 'unit']

## 4. Map biosphere flows

The SimaPro elementary-flow labels are converted to Brightway-compatible names and categories, followed by a few EXIOBASE-specific corrections.


In [7]:
importer.randonneur(
    "SimaPro-9-ecoinvent-3.8-biosphere",
    fields=["name", "context", "unit"],
)
importer.randonneur("simapro-9-ecoinvent-3-context")
importer.normalize_labels_to_brightway_standard()
importer.apply_strategy(bi.strategies.biosphere.drop_unspecified_subcategories)
importer.randonneur("SimaPro-9-ecoinvent-3.8-biosphere-manual-matches")
importer.apply_strategy(tweak3)

2026-09-22 14:18:07.972 | INFO     | randonneur.edges:migrate_edges_with_stored_data:71 - Loaded transformation data SimaPro-9-ecoinvent-3.8-biosphere from registry with following verbs: ['update']
2026-09-22 14:18:07.974 | INFO     | randonneur.generic_transformation:generic_transformation:30 - Can apply the following transformation verbs: ['update']
2026-09-22 14:18:14.207 | INFO     | randonneur.edges:migrate_edges_with_stored_data:71 - Loaded transformation data simapro-9-ecoinvent-3-context from registry with following verbs: ['replace']
2026-09-22 14:18:14.208 | INFO     | randonneur.generic_transformation:generic_transformation:30 - Can apply the following transformation verbs: ['replace']


Applying strategy: normalize_simapro_labels_to_brightway_standard
Applying strategy: drop_unspecified_subcategories


2026-09-22 14:18:21.041 | INFO     | randonneur.edges:migrate_edges_with_stored_data:71 - Loaded transformation data SimaPro-9-ecoinvent-3.8-biosphere-manual-matches from registry with following verbs: ['replace']
2026-09-22 14:18:21.042 | INFO     | randonneur.generic_transformation:generic_transformation:30 - Can apply the following transformation verbs: ['replace']


Applying strategy: tweak3


## 5. Link technosphere and biosphere exchanges

Technosphere exchanges are linked internally within EXIOBASE. Biosphere exchanges are linked to the configured biosphere database.


In [8]:
importer.match_database(
    fields=["name", "unit", "location"]
)

importer.match_database(
    db_name=Config.EXIOBASE_BIOSPHERE_NAME,
    fields=["name", "unit", "categories"],
)
importer.match_database(
    db_name=Config.EXIOBASE_BIOSPHERE_NAME,
    fields=["code"],
)
importer.match_database(
    db_name=Config.EXIOBASE_BIOSPHERE_NAME,
    fields=["name", "categories", "unit"],
)

importer.statistics()

Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Graph statistics for `exiobase3316b2` importer:
38644 graph nodes:
	process: 38644
2720982 graph edges:
	biosphere: 1860685
	technosphere: 821653
	production: 38644
1272997 edges to the following databases:
	exiobase3316b2: 860297
	biosphere3: 412700
82 unique unlinked edges (1447985 total):
	biosphere: 82




(38644, 2720982, 1447985, 0)

## 6. Inspect unlinked exchanges

Check which exchanges could not yet be linked to the configured biosphere database. These remaining flows will be placed in a separate EXIOBASE biosphere database.


In [9]:
pd.DataFrame(importer.unlinked)

,name,context,unit,line_no,amount,uncertainty type,loc,type,categories,original unit before conversion,unit conversion factor
0,Taxes less subsidies on products purchased: Total,"(Economic issues, )",MEUR2011,842,2.422834e+00,0,2.422834e+00,biosphere,"(Economic issues,)",NaN,NaN
1,Other net taxes on production,"(Economic issues, )",MEUR2011,843,9.177310e-01,0,9.177310e-01,biosphere,"(Economic issues,)",NaN,NaN
2,Compensation of employees incl. social contrib...,"(Economic issues, )",MEUR2011,844,3.501562e+00,0,3.501562e+00,biosphere,"(Economic issues,)",NaN,NaN
3,Compensation of employees incl. social contrib...,"(Economic issues, )",MEUR2011,845,1.391057e+01,0,1.391057e+01,biosphere,"(Economic issues,)",NaN,NaN
4,Compensation of employees incl. social contrib...,"(Economic issues, )",MEUR2011,846,3.411892e+00,0,3.411892e+00,biosphere,"(Economic issues,)",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
77,Lead ore,"[natural resource, (unspecified)]",kilogram,969,2.062542e-02,0,2.062542e-05,biosphere,"(natural resource,)",ton,1000.0
78,Platinum,"[natural resource, (unspecified)]",kilogram,970,8.323205e-07,0,8.323205e-10,biosphere,"(natural resource,)",ton,1000.0
79,Silver,"[natural resource, (unspecified)]",kilogram,971,2.075699e-04,0,2.075699e-07,biosphere,"(natural resource,)",ton,1000.0
80,Tin ore,"[natural resource, (unspecified)]",kilogram,972,1.009919e-03,0,1.009919e-06,biosphere,"(natural resource,)",ton,1000.0


## 7. Create an extra biosphere database for remaining flows

Any elementary flows that cannot be linked to the configured biosphere database are stored in a separate EXIOBASE biosphere database.

In [10]:
if Config.EXIOBASE_EXTRA_BIOSPHERE_NAME not in bd.databases:
    importer.create_new_biosphere(Config.EXIOBASE_EXTRA_BIOSPHERE_NAME)

importer.match_database(
    db_name=Config.EXIOBASE_EXTRA_BIOSPHERE_NAME,
    fields=["name", "unit", "categories"],
)

Creating new biosphere database exiobase3316_extra_biosphere_b2 with 82 flows


100%|██████████| 82/82 [00:00<00:00, 7069.83it/s]


14:19:51+0200 [info     ] Vacuuming database            
Applying strategy: link_iterable_by_fields
Applied 1 strategies in 8.52 seconds
Applying strategy: link_iterable_by_fields


## 8. Check that all exchanges are linked

Before writing the database, verify that no exchanges remain unlinked.

In [11]:
pd.DataFrame(importer.unlinked)

""


## 9. Write the EXIOBASE database

This writes the processed datasets to the current Brightway project.

In [12]:
importer.write_database()

14:20:54+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 38644/38644 [02:06<00:00, 305.36it/s]


14:23:04+0200 [info     ] Vacuuming database            
Created database: exiobase3316b2


Brightway2 SQLiteBackend: exiobase3316b2

## 10. Verify the import

In [13]:
bd.databases

Databases dictionary with 6 object(s):
	bafu
	biosphere3
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-consequential
	exiobase3316_extra_biosphere_b2
	exiobase3316b2